<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep — Audio Stem Separator

Separate any audio into stems (vocals, drums, bass, etc.) using state-of-the-art AI models.

---

In [ ]:
#@title 1. Install
#@markdown Clone repo, install deps, yt-dlp, ffmpeg

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
!pip install -q -r /content/vsep/requirements.txt yt-dlp
!apt-get -qq install ffmpeg > /dev/null 2>&1

import os; os.chdir('/content/vsep')
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU — go to Runtime > Change runtime type")
import yt_dlp; print(f"yt-dlp {yt_dlp.version.__version__} ready")

In [ ]:
#@title 2. Get Audio
#@markdown Upload a file OR paste a YouTube/SoundCloud URL

import os, glob
audio_source = "Upload file" #@param ["Upload file", "YouTube / URL"]

if audio_source == "Upload file":
    from google.colab import files
    print("Upload your audio (MP3, WAV, FLAC, OGG, M4A)...")
    uploaded = files.upload()
    if uploaded:
        audio_file = list(uploaded.keys())[0]
    else:
        raise SystemExit("No file uploaded.")
else:
    url = "" #@param {type:"string"}
    if not url.strip():
        raise SystemExit("Paste a URL first.")
    os.makedirs("/content/vsep/audio_input", exist_ok=True)
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": "/content/vsep/audio_input/%(title).80s.%(ext)s",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav", "preferredquality": "0"}],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url.strip()])
    audio_file = sorted(glob.glob("/content/vsep/audio_input/*.wav"))[-1]

print(f"\nAudio: {audio_file} ({os.path.getsize(audio_file)/1e6:.1f} MB)")

## 3. Browse Models

Run this cell to see all available models. Copy the **display name** and paste it in the next cell.

In [ ]:
#@title Browse All Models
#@markdown Filter by category. Copy the display name you want.

MODEL_CATALOG = {
    # ===== VOCALS =====
    "BS Roformer EP 317 SDR 12.98":      "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "BS Roformer EP 368 SDR 12.96":      "model_bs_roformer_ep_368_sdr_12.9628.ckpt",
    "Mel Roformer Viperx SDR 12.61":     "Mel-Roformer-Viperx-1053.ckpt",
    "BS Roformer Viperx 1297":           "BS-Roformer-Viperx-1297.ckpt",
    "BS Roformer Vocals Revive V3e":      "bs_roformer_vocals_revive_v3e_unwa.ckpt",
    "BS Roformer Vocals Revive V2":      "bs_roformer_vocals_revive_v2_unwa.ckpt",
    "BS Roformer Vocals Resurrection":   "bs_roformer_vocals_resurrection_unwa.ckpt",
    "MelBand Vocals FV6 Gabox":          "mel_band_roformer_vocals_fv6_gabox.ckpt",
    "MelBand Vocals FV5 Gabox":          "mel_band_roformer_vocals_fv5_gabox.ckpt",
    "MelBand Vocals V2 Gabox":           "mel_band_roformer_vocals_v2_gabox.ckpt",
    "MelBand Vocals Gabox":              "mel_band_roformer_vocals_gabox.ckpt",
    "MelBand Vocals Becruily":           "mel_band_roformer_vocals_becruily.ckpt",
    "MelBand Kim FT3 Unwa":             "mel_band_roformer_kim_ft3_unwa.ckpt",
    "MelBand Kim FT2 Unwa":             "mel_band_roformer_kim_ft2_unwa.ckpt",
    "MelBand Kim FT Unwa":              "mel_band_roformer_kim_ft_unwa.ckpt",
    "MelBand Big Beta6X":               "melband_roformer_big_beta6x.ckpt",
    "MelBand Big Beta6":                "melband_roformer_big_beta6.ckpt",
    "MelBand Big Beta5e FT":            "melband_roformer_big_beta5e.ckpt",
    "MelBand Big Beta4 FT":             "melband_roformer_big_beta4.ckpt",
    "MelBand Big SYHFT V1":             "MelBandRoformerBigSYHFTV1.ckpt",
    "MelBand SYHFT V3e":                "MelBandRoformerSYHFTV3Epsilon.ckpt",
    "MelBand SYHFT V2":                 "MelBandRoformerSYHFTV2.ckpt",
    "MelBand SYHFT":                    "MelBandRoformerSYHFT.ckpt",
    "MelBand Vocal Fullness Aname":     "mel_band_roformer_vocal_fullness_aname.ckpt",
    "MelBand Bleed Suppressor V1":      "mel_band_roformer_bleed_suppressor_v1.ckpt",
    "MelBand Karaoke V2 Gabox":         "mel_band_roformer_karaoke_gabox_v2.ckpt",
    "MelBand Karaoke Gabox":            "mel_band_roformer_karaoke_gabox.ckpt",
    "MelBand Karaoke Aufr33 Viperx":    "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956_config.yaml",
    "MelBand Karaoke Becruily":         "mel_band_roformer_karaoke_becruily.ckpt",
    "MelBand Denoise Debleed Gabox":    "mel_band_roformer_denoise_debleed_gabox.ckpt",
    "MelBand Denoise Aufr33 Aggr":     "denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768_config.yaml",
    "MelBand Denoise Aufr33":           "denoise_mel_band_roformer_aufr33_sdr_27.9959_config.yaml",
    "MelBand Crowd Viperx":             "mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144_config.yaml",
    "MelBand Aspiration":               "config_aspiration_mel_band_roformer.yaml",
    "MDX23C InstVoc HQ":                "MDX23C-8KFFT-InstVoc_HQ.ckpt",
    "MDX23C InstVoc HQ 2":              "MDX23C-8KFFT-InstVoc_HQ_2.ckpt",
    "MDX Net Voc FT":                   "UVR-MDX-NET-Voc_FT.onnx",
    "MDX Net Main 438":                 "UVR-MDX-NET_Main_438.onnx",
    "MDX Net Main 427":                 "UVR-MDX-NET_Main_427.onnx",
    "MDX Net Main 406":                 "UVR-MDX-NET_Main_406.onnx",
    "MDX Net Main 340":                 "UVR-MDX-NET_Main_340.onnx",
    "MDX Net Inst HQ 5":                "UVR-MDX-NET-Inst_HQ_5.onnx",
    "MDX Net Inst HQ 4":                "UVR-MDX-NET-Inst_HQ_4.onnx",
    "MDX Net Inst HQ 2":                "UVR-MDX-NET-Inst_HQ_2.onnx",
    "MDX Net Inst 1":                   "UVR-MDX-NET-Inst_1.onnx",
    "MDX Net Crowd HQ 1":               "UVR-MDX-NET_Crowd_HQ_1.onnx",
    "Kim Vocal 2":                      "Kim_Vocal_2.onnx",
    "Kim Vocal 1":                      "Kim_Vocal_1.onnx",
    "MDX Net Karaoke 2":                "UVR_MDXNET_KARA_2.onnx",
    "MDX Net Karaoke":                  "UVR_MDXNET_KARA.onnx",

    # ===== INSTRUMENTAL =====
    "MelBand INSTV8N Gabox":            "mel_band_roformer_instrumental_instv8n_gabox.ckpt",
    "MelBand INSTV8 Gabox":             "mel_band_roformer_instrumental_instv8_gabox.ckpt",
    "MelBand INSTV7N Gabox":            "mel_band_roformer_instrumental_instv7n_gabox.ckpt",
    "MelBand INSTV7 Gabox":             "mel_band_roformer_instrumental_instv7_gabox.ckpt",
    "MelBand Instrumental FVX Gabox":   "mel_band_roformer_instrumental_fvx_gabox.ckpt",
    "MelBand Instrumental FV8 Gabox":   "mel_band_roformer_instrumental_fv8_gabox.ckpt",
    "MelBand Instrumental Fullness V4": "mel_band_roformer_instrumental_fullness_noise_v4_gabox.ckpt",
    "MelBand Instrumental Bleedless V3":"mel_band_roformer_instrumental_bleedless_v3_gabox.ckpt",
    "MelBand Instrumental 3 Gabox":     "mel_band_roformer_instrumental_3_gabox.ckpt",
    "MelBand Instrumental 2 Gabox":     "mel_band_roformer_instrumental_2_gabox.ckpt",
    "MelBand Instrumental V2":          "melband_roformer_inst_v2.ckpt",
    "MelBand Inst V1E Plus Unwa":      "melband_roformer_inst_v1e_plus.ckpt",
    "MelBand Inst V1E Unwa":           "melband_roformer_inst_v1e.ckpt",
    "MelBand Inst V1 Plus Unwa":       "melband_roformer_inst_v1_plus.ckpt",
    "MelBand Inst V1 Unwa":            "melband_roformer_inst_v1.ckpt",
    "MelBand Instrumental Gabox":       "mel_band_roformer_instrumental_gabox.ckpt",
    "MelBand Instrumental Becruily":    "mel_band_roformer_instrumental_becruily.ckpt",
    "BS Roformer Instrumental Resurrect":"bs_roformer_instrumental_resurrection_unwa.ckpt",
    "Kim Instrumental":                 "Kim_Inst.onnx",

    # ===== MULTI-STEM =====
    "Demucs v4 Fine Tuned":             "htdemucs_ft.yaml",
    "Demucs v4 6 Stem":                 "htdemucs_6s.yaml",
    "Demucs v4 HT":                     "htdemucs.yaml",
    "Demucs v4 MMI":                    "hdemucs_mmi.yaml",
    "BS Roformer Male-Female Sucial":   "config_chorus_male_female_bs_roformer.yaml",

    # ===== REVERB / ECHO REMOVAL =====
    "DeReverb Echo Fused":              "dereverb_echo_mbr_fused.ckpt",
    "DeReverb Echo Super Big":          "dereverb_super_big_mbr_ep_346.ckpt",
    "DeReverb Echo Big":                "dereverb_big_mbr_ep_362.ckpt",
    "MelBand DeReverb Echo V2 Sucial":  "config_dereverb_echo_mel_band_roformer_v2.yaml",
    "MelBand DeReverb Echo Sucial":     "config_dereverb-echo_mel_band_roformer.yaml",
    "MelBand DeReverb Less Aggressive": "dereverb_mel_band_roformer_mono_anvuew.ckpt",
    "MelBand DeReverb Anvuew":          "dereverb_mel_band_roformer_anvuew.yaml",
    "BS Roformer DeReverb":             "deverb_bs_roformer_8_384dim_10depth.ckpt",
    "MDX23C DeReverb Aufr33":           "MDX23C-De-Reverb-aufr33-jarredou.ckpt",
    "Reverb HQ FoxJoy ONNX":            "Reverb_HQ_By_FoxJoy.onnx",
    "VR DeReverb Aufr33":               "UVR-De-Reverb-aufr33-jarredou.pth",
    "VR DeEcho DeReverb":               "UVR-DeEcho-DeReverb.pth",
    "VR DeEcho Aggressive":             "UVR-De-Echo-Aggressive.pth",
    "VR DeEcho Normal":                 "UVR-De-Echo-Normal.pth",

    # ===== DRUMS / BASS / OTHER =====
    "MDX23C DrumSep Aufr33":            "MDX23C-DrumSep-aufr33-jarredou.ckpt",
    "Kuielab B Drums":                  "kuielab_b_drums.onnx",
    "Kuielab A Drums":                  "kuielab_a_drums.onnx",
    "Kuielab B Bass":                   "kuielab_b_bass.onnx",
    "Kuielab A Bass":                   "kuielab_a_bass.onnx",
    "Kuielab B Other":                  "kuielab_b_other.onnx",
    "Kuielab A Other":                  "kuielab_a_other.onnx",
    "Kuielab B Vocals":                 "kuielab_b_vocals.onnx",
    "Kuielab A Vocals":                 "kuielab_a_vocals.onnx",

    # ===== DENOISE / CLEAN =====
    "VR DeNoise":                       "UVR-DeNoise.pth",
    "VR DeNoise Lite":                  "UVR-DeNoise-Lite.pth",
    "MDX23C D1581":                     "MDX23C_D1581.ckpt",

    # ===== LEGACY / CLASSIC =====
    "MDX Net Main 9703":                "UVR_MDXNET_1_9703.onnx",
    "MDX Net Main 9682":                "UVR_MDXNET_2_9682.onnx",
    "5 HP UVR":                         "5_HP-Karaoke-UVR.pth",
    "2 HP UVR":                         "2_HP-UVR.pth",
    "MGM Main v4":                      "MGM_MAIN_v4.pth",
    "MGM Highend v4":                   "MGM_HIGHEND_v4.pth",
    "MGM Lowend A v4":                  "MGM_LOWEND_A_v4.pth",
    "MGM Lowend B v4":                  "MGM_LOWEND_B_v4.pth",
}

CATEGORIES = {
    "Vocals": lambda d: all(k in d for k in ["Roformer", "Vocals", "MDX", "Kim", "MelBand", "Viperx", "InstVoc", "Voc FT", "Main 4", "Karaoke", "KARA", "Crowd", "Denoise", "Aspiration", "Bleed", "Fullness"]),
}
# Simple keyword-based category filter
cat_filter = "Vocals" #@param ["All", "Vocals", "Instrumental", "Multi-Stem", "Reverb/Echo", "Drums/Bass/Other", "Denoise", "Legacy"]

keywords = {
    "Vocals":         ["Vocal", "Vocals", "InstVoc", "Voc FT", "Karaoke", "KARA", "Crowd", "Denoise", "Aspiration", "Bleed", "Fullness", "Male-Female"],
    "Instrumental":   ["Inst", "Instrumental", "Kim Inst"],
    "Multi-Stem":     ["Demucs", "Male-Female"],
    "Reverb/Echo":    ["Reverb", "Echo", "DeReverb", "DeEcho"],
    "Drums/Bass/Other":["Drum", "Bass", "Other", "Kuielab"],
    "Denoise":        ["DeNoise"],
    "Legacy":         ["MDX23C D1581", "MDX Net Main 9", "HP UVR", "MGM"],
}

if cat_filter == "All":
    shown = MODEL_CATALOG
else:
    kws = keywords.get(cat_filter, [])
    shown = {k: v for k, v in MODEL_CATALOG.items() if any(kw.lower() in k.lower() for kw in kws)}

print(f"\n{len(shown)} models in [{cat_filter}]\n")
print(f"{'#':<4} {'Display Name':<42} {'Filename':<50}")
print("=" * 100)
for i, (name, fn) in enumerate(shown.items(), 1):
    print(f"{i:<4} {name:<42} {fn:<50}")
print(f"\nTotal catalog: {len(MODEL_CATALOG)} models")
print("Copy the display name and paste in the next cell.")

In [ ]:
#@title 4. Select Model
#@markdown Paste the display name from the browser above, or any model filename

model = "BS Roformer EP 317 SDR 12.98" #@param {type:"string"}

if model in MODEL_CATALOG:
    selected_model = MODEL_CATALOG[model]
    print(f"Model: {model}")
    print(f"File:  {selected_model}")
elif model.endswith((".ckpt", ".onnx", ".pth", ".yaml")):
    selected_model = model
    print(f"Custom model file: {model}")
else:
    # fuzzy search
    matches = [k for k in MODEL_CATALOG if model.lower() in k.lower()]
    if matches:
        selected_model = MODEL_CATALOG[matches[0]]
        print(f"Did you mean: {matches[0]}")
        print(f"File:  {selected_model}")
    else:
        raise SystemExit(f"Model not found: {model}. Run the browser cell to see available models.")

In [ ]:
#@title 5. Separate
#@markdown Run the AI separation on your audio

from separator import Separator
import config.variables as cfg

cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print(f"Separating {audio_file} with {selected_model}...")
print("(downloading model on first use, cached after)")

separator = Separator(
    model_file_dir="/content/vsep/models",
    output_dir="/content/vsep/output",
    output_format="WAV",
    use_soundfile=True,
)
separator.load_model(model_filename=selected_model)
output_files = separator.separate(audio_file)

print(f"\nDone! {len(output_files)} stem(s):")
for f in output_files:
    print(f"  {f}")

In [ ]:
#@title 6. Listen & Download
#@markdown Play the results and download to your device

import glob
from IPython.display import Audio, display
from google.colab import files as dl

stems = sorted(glob.glob("/content/vsep/output/*.*"))
if not stems:
    raise SystemExit("No stems found. Run the separation cell first.")

for i, path in enumerate(stems, 1):
    name = os.path.basename(path)
    size = os.path.getsize(path) / 1e6
    print(f"{i}. {name} ({size:.1f} MB)")
    display(Audio(path))

print("\n--- Downloading all stems ---")
for path in stems:
    dl.download(path)